# Modular Retrieval-Augmented Generation (RAG) System

This Jupyter Notebook implements a clean, modular, and educational **Retrieval-Augmented Generation (RAG)** pipeline from scratch. The system is designed to load, clean, chunk, and index a PDF document, and answer user queries using context retrieved from the document and **Google Gemini** as the Large Language Model.

---

### Project Objective
The main goal is to build a RAG pipeline that answers questions from a custom PDF document using semantic similarity retrieval to supply context to Google Gemini, avoiding high-level abstractions (like full LangChain pipelines) to illustrate RAG's internal mechanics.

### Key Concepts Explained

1. **Retrieval**: The process of searching a collection of documents to find the most relevant information matching a user's query. Rather than searching text directly, we convert documents and queries into vectors and search by semantic similarity.
2. **Embeddings**: Numerical vector representations of text where geometrically close vectors represent semantically similar concepts. This allows machines to understand and compare word/sentence meanings.
3. **Vector Database**: A specialized database designed to store, manage, and query high-dimensional vector embeddings efficiently. In this project, we use **FAISS (Facebook AI Similarity Search)**.
4. **Chunking**: The practice of breaking a large document into smaller, manageable text segments ("chunks"). This ensures that retrieved context is highly focused and fits within the LLM's context window.
5. **Generation**: The phase where the LLM (Google Gemini) takes the user's question along with the retrieved chunks (context) and generates a coherent, factually grounded answer.


## System Architecture

The following ASCII diagram illustrates the end-to-end data flow of our modular RAG system:

```text
+------------------+      +-------------------+      +-------------------------+
|                  |      |                   |      |                         |
|  PDF Document    | ---> | PyPDF Text Extract| ---> | Text Cleaning & Chunking|
|                  |      |                   |      | (RecursiveCharacter)    |
+------------------+      +-------------------+      +-------------------------+
                                                                  |
                                                                  v
+------------------+      +-------------------+      +-------------------------+
|                  |      |                   |      |  Sentence Transformers  |
|  FAISS Index     | <--- |   FAISS Index     | <--- |    Embedding Model      |
|  (Vector DB)     |      |   Construction    |      |    (all-MiniLM-L6-v2)   |
+------------------+      +-------------------+      +-------------------------+
         |
         v
+------------------+      +-------------------+      +-------------------------+
|                  |      |                   |      |                         |
|  User Query      | ---> | Query Embedding   | ---> | Similarity Search       |
|                  |      | (Sentence Trans)  |      | (FAISS Index Lookup)    |
+------------------+      +-------------------+      +-------------------------+
                                                                  |
                                                                  v
+------------------+      +-------------------+      +-------------------------+
|                  |      |                   |      |                         |
|  Generated       | <--- | Gemini LLM        | <--- | Context + Query Prompt  |
|  Answer (RAG)    |      | (gemini-1.5-flash)|      |                         |
+------------------+      +-------------------+      +-------------------------+
```


### Step 1: Install Dependencies
First, we install all the required Python libraries. This includes `pypdf` for reading PDFs, `sentence-transformers` for creating text embeddings, `faiss-cpu` for similarity search, `google-generativeai` to access Gemini, and `langchain-text-splitters` specifically for character-based text splitting.


In [ ]:
# Install dependencies (uncomment if running in a new environment)
# !pip install pypdf sentence-transformers faiss-cpu google-generativeai langchain-text-splitters


### Step 2: Import Libraries
Next, we import standard and third-party libraries needed for our pipeline.


In [ ]:
import os
import re
import getpass
import numpy as np
import pypdf
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import google.generativeai as genai

print("All libraries imported successfully!")


### Step 3: Configure Gemini API
To interact with Google Gemini, we need an API Key. We'll set this up securely using `getpass` to hide input typing. 

**Important Security Note**: To avoid exposing your Gemini API Key, do not hardcode it in this notebook, and remember to clear the cell output of the API key block before saving or submitting your notebook.


In [ ]:
# SECURITY WARNING: Clear cell outputs before submitting this notebook to avoid exposing API keys.
def configure_gemini_api(api_key=None):
    """
    Configures the Google Generative AI client with the provided or environment-stored API key.
    """
    if not api_key:
        api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            print("GEMINI_API_KEY environment variable not found.")
            # getpass masks the API key while typing
            api_key = getpass.getpass("Please enter your Google Gemini API Key:")
            
    if not api_key.strip():
        raise ValueError("API Key cannot be empty.")
        
    genai.configure(api_key=api_key)
    print("Gemini API configured successfully!")
    return api_key

# Run configuration
try:
    gemini_api_key = configure_gemini_api()
except Exception as e:
    print(f"Configuration failed: {e}")


### Step 4 & 5: Load PDF & Extract Text
We define `load_pdf` to load the PDF document using `pypdf.PdfReader` and extract text page-by-page. Basic exception handling ensures that missing files or empty documents raise descriptive errors.


In [ ]:
def load_pdf(pdf_path: str) -> list[str]:
    """
    Loads a PDF file from the local path and extracts text page by page.
    
    Args:
        pdf_path (str): Absolute or relative path to the PDF document.
        
    Returns:
        list[str]: A list of strings, where each element corresponds to a page's text.
        
    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file contains no text content.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF document not found at: {pdf_path}")
        
    reader = pypdf.PdfReader(pdf_path)
    pages_text = []
    
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages_text.append(text)
        else:
            pages_text.append("")  # Append empty string if page has no extractable text
            
    # Check if we got any text at all
    if not pages_text or all(len(page.strip()) == 0 for page in pages_text):
        raise ValueError("The PDF document is empty or contains no extractable text.")
        
    print(f"Successfully loaded PDF '{pdf_path}' with {len(pages_text)} pages.")
    return pages_text


### Step 6: Clean Extracted Text
Text extracted from PDFs often contains extra spaces, tabs, or weird formatting. We'll define a function `clean_text` to normalize horizontal spacing and limit consecutive newlines while preserving document paragraph structure.


In [ ]:
def clean_text(pages_text: list[str]) -> list[str]:
    """
    Cleans the raw text of each page by removing excess white space and normalizing formatting.
    
    Args:
        pages_text (list[str]): Raw extracted text per page.
        
    Returns:
        list[str]: Cleaned text per page.
    """
    cleaned_pages = []
    for text in pages_text:
        # Collapse multiple spaces or tabs into a single space
        text = re.sub(r'[ \t]+', ' ', text)
        # Collapse three or more consecutive newlines into double newlines
        text = re.sub(r'\n{3,}', '\n\n', text)
        cleaned_pages.append(text.strip())
    print("Text cleaning completed.")
    return cleaned_pages


### Step 7: Text Chunking
To ensure our text segments fit inside the model's context window and stay highly semantically focused, we split the document into overlapping chunks. Using `RecursiveCharacterTextSplitter` from LangChain allows us to split cleanly on paragraphs or sentence boundaries first. We'll retain the source page number of each chunk as metadata to trace where retrieved information came from.


In [ ]:
def split_text(cleaned_pages: list[str], chunk_size: int = 500, chunk_overlap: int = 50) -> list[dict]:
    """
    Splits text from pages into smaller overlapping chunks, preserving page numbers as metadata.
    
    Args:
        cleaned_pages (list[str]): Cleaned text of pages.
        chunk_size (int): Max character count of each chunk.
        chunk_overlap (int): Overlap character count between consecutive chunks.
        
    Returns:
        list[dict]: A list of chunks, where each chunk is represented as:
                    {"text": str, "page": int}
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    chunks = []
    for page_idx, page_text in enumerate(cleaned_pages):
        page_num = page_idx + 1
        if not page_text.strip():
            continue
            
        page_chunks = splitter.split_text(page_text)
        for chunk in page_chunks:
            chunks.append({
                "text": chunk,
                "page": page_num
            })
            
    print(f"Created {len(chunks)} text chunks (size={chunk_size}, overlap={chunk_overlap}).")
    return chunks


### Step 8: Create Embeddings
We convert the text chunks into mathematical vector representations using Sentence Transformers. The pre-trained model `all-MiniLM-L6-v2` is a lightweight, high-performance model that embeds text into 384-dimensional vectors.


In [ ]:
def create_embeddings(chunks: list[dict], model_name: str = "all-MiniLM-L6-v2"):
    """
    Generates embedding vectors for all text chunks using a SentenceTransformer model.
    
    Args:
        chunks (list[dict]): Text chunks with metadata.
        model_name (str): SentenceTransformer pre-trained model name.
        
    Returns:
        tuple: (embeddings numpy.ndarray, loaded transformer model object)
    """
    print(f"Loading embedding model '{model_name}'...")
    model = SentenceTransformer(model_name)
    
    texts = [chunk["text"] for chunk in chunks]
    print(f"Generating embeddings for {len(texts)} chunks...")
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    
    print(f"Finished generating embeddings. Dimensions: {embeddings.shape[1]}")
    return embeddings, model


### Step 9: Build FAISS Vector Index
We store the chunk vectors in a FAISS vector database. Since we want to use **cosine similarity**, we normalize our embeddings to unit length and use an **Inner Product (IP)** flat index. This yields the exact same ranking as cosine similarity search.


In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """
    Constructs a FAISS Flat IP index, normalizing embeddings to compute cosine similarity scores.
    
    Args:
        embeddings (np.ndarray): Chunks embedding vectors.
        
    Returns:
        faiss.IndexFlatIP: The FAISS index with added vectors.
    """
    # Ensure float32 representation
    embeddings_f32 = np.array(embeddings, dtype=np.float32)
    
    # Normalize vectors for cosine similarity (norm(v) = 1)
    faiss.normalize_L2(embeddings_f32)
    
    dimension = embeddings_f32.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings_f32)
    
    print(f"FAISS index built successfully containing {index.ntotal} vectors.")
    return index


### Step 10 & 11: Context Retrieval
To search, we convert the user's query into an embedding, normalize it, and perform a similarity search on the FAISS index. The index returns the top `k` indices and their similarity scores.


In [ ]:
def retrieve_documents(query: str, index: faiss.IndexFlatIP, chunks: list[dict], model: SentenceTransformer, k: int = 3) -> list[dict]:
    """
    Retrieves the top-k most semantically similar text chunks for a given query.
    
    Args:
        query (str): The user query.
        index (faiss.IndexFlatIP): Built FAISS index.
        chunks (list[dict]): Core document chunks list.
        model (SentenceTransformer): Model used to embed query.
        k (int): Number of chunks to retrieve.
        
    Returns:
        list[dict]: List of retrieved chunks with matched texts, page numbers, and cosine similarity scores.
    """
    if not query.strip():
        raise ValueError("The query question is empty.")
        
    # Embed the query
    query_emb = model.encode([query], show_progress_bar=False, convert_to_numpy=True)
    query_emb_f32 = np.array(query_emb, dtype=np.float32)
    
    # Normalize query vector
    faiss.normalize_L2(query_emb_f32)
    
    # Query FAISS
    scores, indices = index.search(query_emb_f32, k)
    
    retrieved = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(chunks):
            continue
        retrieved.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "score": float(score)
        })
        
    return retrieved


### Step 12, 13 & 14: Prompt Assembly & Answer Generation
We assemble the context along with the user's query into a prompt. This prompt explicitly instructs the LLM to restrict its answers to the retrieved text. We then submit this prompt to Gemini for answer generation.


In [ ]:
def build_prompt(query: str, retrieved_results: list[dict]) -> str:
    """
    Assembles context segments and query into a grounded prompt instructions template.
    """
    context_blocks = []
    for i, res in enumerate(retrieved_results):
        context_blocks.append(
            f"--- Context Segment {i+1} (Source: Page {res['page']}, Similarity: {res['score']:.4f}) ---\n"
            f"{res['text']}"
        )
        
    context_str = "\n\n".join(context_blocks)
    
    prompt = f"""You are an intelligent AI assistant answering questions based on the provided document context.

Use ONLY the retrieved context below to answer the user's question. If the context does not contain the answer or is not relevant, honestly state that the answer cannot be found in the document. Do not make up or hallucinate information.

Retrieved Context:
{context_str}

User Question: {query}

Answer:"""
    return prompt

def generate_answer(prompt: str, api_key: str = None) -> str:
    """
    Sends the prompt to Google Gemini model to generate a response.
    """
    if api_key:
        genai.configure(api_key=api_key)
        
    try:
        model = genai.GenerativeModel('gemini-2.5-flash')
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        raise RuntimeError(f"Gemini API generation failed: {e}")


## Main Pipeline Orchestrator
We define a coordinating pipeline orchestrator `run_rag_pipeline` that wraps the end-to-end execution. (Note: In the final deployment, we partition this process into Phase 1 Document Preprocessing and Phase 2 Inference for improved efficiency).


In [ ]:
def run_rag_pipeline(pdf_path: str, query: str, api_key: str, k: int = 3, chunk_size: int = 500, chunk_overlap: int = 50):
    """
    Executes the full RAG pipeline (loading, indexing, and generating) in one coordinated run.
    """
    if not query or not query.strip():
        print("[ERROR]: Question is empty.")
        return
        
    try:
        raw_pages = load_pdf(pdf_path)
        cleaned_pages = clean_text(raw_pages)
        chunks = split_text(cleaned_pages, chunk_size, chunk_overlap)
        embeddings, model = create_embeddings(chunks)
        index = build_faiss_index(embeddings)
        retrieved_results = retrieve_documents(query, index, chunks, model, k)
        prompt = build_prompt(query, retrieved_results)
        answer = generate_answer(prompt, api_key)
        
        # Display outputs
        print("\n" + "="*80)
        print("                         Retrieved Context")
        print("="*80)
        for i, chunk in enumerate(retrieved_results):
            print(f"\n[Chunk {i+1}] (Source: Page {chunk['page']}, Similarity Score: {chunk['score']:.4f})")
            print(f"\"{chunk['text'].strip()}\"")
            print("-" * 60)
            
        print("\n" + "="*80)
        print("                         Generated Answer")
        print("="*80)
        print(answer.strip())
        print("="*80 + "\n")
        
    except Exception as e:
        print(f"\n[ERROR]: Pipeline failed: {e}")


## Phase 1: Document Processing (Executed Once)
In this phase, we prompt the user for the PDF file path, load the document, clean the text, split it into overlapping chunks, generate embeddings, and build the FAISS index. 

This preprocessing stage is executed **only once** per document, drastically improving query latency since the database is built ahead of inference time.


In [ ]:
# Phase 1: Document Processing Setup

# Prompt user for PDF file path instead of hardcoding
pdf_file_path = input("Enter PDF path (e.g., Week7_Project.pdf): ").strip()

# Global placeholders for preprocessed data
chunks = None
embeddings = None
embedding_model = None
faiss_index = None

if not pdf_file_path:
    print("ERROR: PDF path cannot be empty.")
elif not os.path.exists(pdf_file_path):
    print(f"ERROR: PDF file '{pdf_file_path}' does not exist. Please check the path and try again.")
else:
    try:
        print("\n--- Starting Phase 1: Document Processing ---")
        
        # 1. Load and extract text
        raw_pages = load_pdf(pdf_file_path)
        
        # 2. Clean text
        cleaned_pages = clean_text(raw_pages)
        
        # 3. Split into overlapping chunks
        chunks = split_text(cleaned_pages, chunk_size=500, chunk_overlap=50)
        
        # 4. Generate embeddings (using Sentence Transformers)
        embeddings, embedding_model = create_embeddings(chunks)
        
        # 5. Build local FAISS index
        faiss_index = build_faiss_index(embeddings)
        
        print("\n--- Phase 1: Document Processing Completed Successfully! ---")
        print(f"Total document chunks indexed: {len(chunks)}")
        
    except Exception as e:
        print(f"\n[ERROR] Phase 1 Document Processing failed: {e}")


## Phase 2: Interactive Question Answering (Inference Loop)
Once the document is processed and the FAISS index is built, we enter the interactive chat loop. 

For each question asked by the user, the system only:
1. Converts the query into an embedding.
2. Retrieves the top-k matched chunks from the pre-built FAISS index.
3. Builds the grounding prompt instructions.
4. Calls Gemini to generate the answer.

This separation mimics real-world production setups where data preprocessing and indexing are decoupled from inference services. 

Type **`exit`** to end the chat session.


In [ ]:
# Phase 2: Interactive Q&A loop

# Verify prerequisites before starting loop
if not pdf_file_path or not os.path.exists(pdf_file_path):
    print("ERROR: Please successfully run Phase 1 (Document Processing) first.")
elif 'gemini_api_key' not in globals() or not gemini_api_key:
    print("ERROR: Gemini API Key is not configured. Please run Step 3 first.")
elif faiss_index is None or chunks is None or embedding_model is None:
    print("ERROR: Preprocessed document data is missing. Please run Phase 1 successfully first.")
else:
    print("="*80)
    print(f"Successfully loaded index for '{pdf_file_path}'")
    print("Initializing interactive RAG Chatbot loop. Type 'exit' to quit.")
    print("="*80)
    
    # Run interactive loop
    while True:
        try:
            # Ask the user for a question
            user_question = input("Enter your question: ")
            
            # Check for exit condition
            if user_question.strip().lower() == 'exit':
                print("Exiting interactive RAG loop. Goodbye!")
                break
                
            # Skip empty questions
            if not user_question.strip():
                print("Question cannot be empty. Please try again.\n")
                continue
                
            # Execute inference steps:
            # 1. Retrieve matching chunks from the pre-built index
            retrieved_results = retrieve_documents(
                query=user_question,
                index=faiss_index,
                chunks=chunks,
                model=embedding_model,
                k=3
            )
            
            # 2. Construct grounded prompt
            prompt = build_prompt(user_question, retrieved_results)
            
            # 3. Generate answer using Gemini
            answer = generate_answer(prompt, gemini_api_key)
            
            # 4. Display Outputs (separated and including page source and similarity score)
            print("\n" + "="*80)
            print("                         Retrieved Context")
            print("="*80)
            for i, chunk in enumerate(retrieved_results):
                print(f"\n[Chunk {i+1}] (Source: Page {chunk['page']}, Similarity Score: {chunk['score']:.4f})")
                print(f"\"{chunk['text'].strip()}\"")
                print("-" * 60)
                
            print("\n" + "="*80)
            print("                         Generated Answer")
            print("="*80)
            print(answer.strip())
            print("="*80 + "\n")
            
        except KeyboardInterrupt:
            print("\nExiting interactive RAG loop. Goodbye!")
            break
        except Exception as e:
            print(f"An error occurred: {e}\n")


## Setup Instructions

1. Install the required libraries.
2. Generate a Google Gemini API Key.
3. Enter the API key when prompted.
4. Place the PDF file in the same directory as the notebook.
5. Run all cells from top to bottom.

## Conclusion & Project Summary

This project demonstrates a fully functional, local Retrieval-Augmented Generation (RAG) system using a modular codebase:

1. **PDF Loading**: We load raw pages of the selected PDF using `pypdf.PdfReader` and clean the extracted texts to eliminate redundant whitespace.
2. **Chunking**: The document text is divided into overlapping blocks using `RecursiveCharacterTextSplitter` from `langchain-text-splitters` while preserving the source page number of each block.
3. **Embeddings**: We encode the text blocks into dense 384-dimensional semantic vectors using `SentenceTransformer` and the `all-MiniLM-L6-v2` model.
4. **FAISS Retrieval**: We store vector representations in a local FAISS Inner Product index. Standardizing vector magnitudes allows us to rank and search documents using exact **Cosine Similarity**.
5. **Gemini Answer Generation**: We build an instructional grounding prompt combining the retrieved context chunks and the user's question, passing it to `gemini-1.5-flash` to get an answer rooted strictly in the document text.

By implementing these steps explicitly, this project illustrates the internal mechanics of a RAG pipeline, providing a clean and professional foundation for building domain-specific document assistants.
